In [1]:
!pip install -q -U "bitsandbytes>=0.46.1" peft accelerate

In [2]:
import bitsandbytes, transformers.utils
print("bnb:", bitsandbytes.__version__)
print("transformers sees it:", transformers.utils.is_bitsandbytes_available())

bnb: 0.50.2
transformers sees it: True


In [3]:
# =============================================================================
# KAGGLE - 2-SLM GUIDED, FULL GSM8K TEST (1319), BATCHED
# =============================================================================
# Self-contained. Does NOT need sgft_project.json, Cells 1-4, or Unsloth.
# Plain transformers + peft + bitsandbytes, so no compute-capability or
# monkey-patching issues.
#
# Sequential would be 9.8h. Batched (bs=8) is ~2.2h, well inside Kaggle's 12h.
#
# SETUP (see the steps in chat):
#   Accelerator = GPU T4 x2 (or P100)   Internet = ON   adapter added as Dataset
#
# Writes incrementally to /kaggle/working/twoslm_full_1319.csv with fsync after
# every batch, and resumes on restart. If the session dies at hour 9 you lose
# nothing.
# =============================================================================

import os, re, csv, gc, json, math, time, glob, traceback
from collections import Counter
from typing import Any, Dict, List, Optional, Sequence, Tuple

import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_dataset

# =============================================================================
# CONFIG
# =============================================================================
# Point this at wherever your adapter landed after adding the Kaggle Dataset.
# It is auto-detected below; override only if detection fails.
ADAPTER_DIR   = None
PLANNER_BASE  = "Qwen/Qwen3-4B"              # adapter's base; downloaded from HF
EXECUTOR_MODEL = "Qwen/Qwen2.5-Math-1.5B"

N_PROBLEMS   = 1319        # full test set
BATCH_SIZE   = 8           # drops automatically on OOM
RUN_BASELINE =  True      # set True to also get Phase-B-alone in the same pass
OUT_CSV      = "/kaggle/working/twoslm_full_1319.csv"

PLANNER_MAX_NEW  = 192
EXECUTOR_MAX_NEW = 640
REPETITION_PENALTY = 1.15
EOS_LITERAL = "<|endoftext|>"

# --- prompts: EXACTLY the fixed versions --------------------------------------
# Planner ends "Plan:\n" - the trailing newline is required. Qwen merges ':' with
# a following newline into one ':\n' token, so without it the model continues
# from a token it never saw in training and emits prose or nothing.
PROMPT_PLANNER = ("Instruction: Provide a semantic plan without math.\n"
                  "Question: {question}\n"
                  "Plan:\n")
# Executor uses Qwen-Math's native \boxed{} format, NOT "#### [number]".
PROMPT_EXECUTOR = ("Solve the following math problem. A step-by-step plan is provided; follow it.\n"
                   "Reason step by step, and put your final answer within \\boxed{}.\n\n"
                   "Problem: {question}\n\n"
                   "Plan:\n{plan}\n\n"
                   "Solution:")
PROMPT_BASELINE = ("Solve the following math problem.\n"
                   "Reason step by step, and put your final answer within \\boxed{}.\n\n"
                   "Problem: {question}\n\n"
                   "Solution:")

assert "\\boxed" in PROMPT_EXECUTOR and "\\boxed" in PROMPT_BASELINE
assert "####" not in PROMPT_EXECUTOR
assert PROMPT_PLANNER.endswith("Plan:\n")


def find_adapter() -> str:
    if ADAPTER_DIR and os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")):
        return ADAPTER_DIR
    hits = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    if not hits:
        raise FileNotFoundError(
            "No adapter_config.json under /kaggle/input/. Add your phaseA_planner "
            "folder as a Kaggle Dataset (see setup steps)."
        )
    d = os.path.dirname(hits[0])
    print(f"[find] adapter -> {d}")
    return d


ADAPTER = find_adapter()
print(f"[cfg] planner base : {PLANNER_BASE}")
print(f"[cfg] executor     : {EXECUTOR_MODEL}")
print(f"[cfg] n            : {N_PROBLEMS}   batch={BATCH_SIZE}   baseline={RUN_BASELINE}")


# =============================================================================
# MODEL LOADING
# =============================================================================
def bnb():
    return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=torch.float16,
                              bnb_4bit_use_double_quant=True)


def load_lm(model_id, device_map="auto"):
    kw = {"device_map": device_map, "low_cpu_mem_usage": True, "trust_remote_code": True}
    try:
        cfg = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
        pre = getattr(cfg, "quantization_config", None) is not None
    except Exception:
        pre = False
    if not pre:
        kw["quantization_config"] = bnb()
    for key in ["torch_dtype", "dtype"]:
        try:
            m = AutoModelForCausalLM.from_pretrained(model_id, **{**kw, key: torch.float16})
            m.eval()
            return m
        except TypeError as e:
            if "dtype" not in str(e):
                raise
    raise RuntimeError(f"could not load {model_id}")


def prep_tok(tok):
    eid = tok.convert_tokens_to_ids(EOS_LITERAL)
    if isinstance(eid, int) and eid >= 0:
        tok.eos_token = EOS_LITERAL
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"        # REQUIRED for batched decoder-only generation
    return tok


def stop_ids(tok):
    s = set()
    for t in [EOS_LITERAL, "<|im_end|>", tok.eos_token]:
        if t:
            i = tok.convert_tokens_to_ids(t)
            if isinstance(i, int) and i >= 0:
                s.add(i)
    return sorted(s) or [tok.eos_token_id]


print("\n[load] planner base ...")
planner = load_lm(PLANNER_BASE)
print(f"[load] planner adapter <- {ADAPTER}")
planner = PeftModel.from_pretrained(planner, ADAPTER, is_trainable=False).eval()
try:
    ptok = AutoTokenizer.from_pretrained(ADAPTER, trust_remote_code=True)
except Exception:
    ptok = AutoTokenizer.from_pretrained(PLANNER_BASE, trust_remote_code=True)
ptok = prep_tok(ptok)

print(f"[load] executor <- {EXECUTOR_MODEL}")
executor = load_lm(EXECUTOR_MODEL)
etok = prep_tok(AutoTokenizer.from_pretrained(EXECUTOR_MODEL, trust_remote_code=True))

for m in (planner, executor):
    if getattr(m, "generation_config", None) is not None:
        m.generation_config.max_length = None

print(f"[vram] {torch.cuda.memory_allocated()/1024**3:.2f} GB allocated")


# =============================================================================
# BATCHED GENERATION (with OOM backoff)
# =============================================================================
@torch.no_grad()
def gen_batch(model, tok, prompts: List[str], max_new: int,
              rep_pen: float = 1.0) -> List[str]:
    enc = tok(prompts, return_tensors="pt", padding=True,
              add_special_tokens=False).to(model.device)
    plen = enc["input_ids"].shape[1]
    out = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                         temperature=None, top_p=None, top_k=None,
                         repetition_penalty=rep_pen,
                         eos_token_id=stop_ids(tok), pad_token_id=tok.pad_token_id)
    # left padding => all rows aligned right, so slicing at plen is correct
    return [t.strip() for t in tok.batch_decode(out[:, plen:], skip_special_tokens=True)]


def gen_safe(model, tok, prompts, max_new, rep_pen=1.0):
    """Halve the batch on CUDA OOM rather than losing the run."""
    try:
        return gen_batch(model, tok, prompts, max_new, rep_pen)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(prompts) == 1:
            print("   [oom] even batch=1 failed; returning empty")
            return [""]
        h = len(prompts) // 2
        print(f"   [oom] splitting batch {len(prompts)} -> {h}+{len(prompts)-h}")
        return (gen_safe(model, tok, prompts[:h], max_new, rep_pen) +
                gen_safe(model, tok, prompts[h:], max_new, rep_pen))


# =============================================================================
# PLAN CLEANING + ANSWER EXTRACTION  (same logic as your notebook)
# =============================================================================
THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)
STEP_RE = re.compile(r"^[ \t]*Step[ \t]*(\d+)[ \t]*(?:\[[ \t]*([^\]]{1,32}?)[ \t]*\])?[ \t]*:[ \t]*(.*?)[ \t]*$",
                     re.MULTILINE | re.IGNORECASE)
STEP_SPLIT_RE = re.compile(r"[ \t]*(Step[ \t]*\d+[ \t]*(?:\[|:))", re.IGNORECASE)
ARITH_RE = re.compile(r"\d[ \t]*(?:[+*/×÷])[ \t]*\d|\d[ \t]+-[ \t]+\d|=[ \t]*\$?\d|<<[^>]*>>")
PLAN_STOPS = ("\nQuestion:", "\nInstruction:", "Instruction:")
EXEC_STOPS = ("\nProblem:", "\nQuestion:", "\nPlan:", "\nSolution:")
OPS = {"ADD","SUBTRACT","MULTIPLY","DIVIDE","CONVERT","RATIO","COMPARE","COUNT"}
MAX_STEPS = 9
FALLBACK_PLAN = ("Step 1 [COUNT]: Identify the quantities given in the problem.\n"
                 "Step 2 [COMPARE]: Determine the relationship between them.\n"
                 "Step 3 [ADD]: Compute the requested quantity.")


def cut(text, markers):
    c = len(text)
    for m in markers:
        p = text.find(m)
        if p != -1:
            c = min(c, p)
    return text[:c].strip()


def sanitise_plan(raw):
    t = THINK_RE.sub("", raw or "")
    t = re.sub(r"^\s*<think>", "", t, flags=re.IGNORECASE)
    t = cut(t, PLAN_STOPS)
    t = re.sub(r"^\s*Plan\s*:\s*", "", t, flags=re.IGNORECASE)
    t = STEP_SPLIT_RE.sub(r"\n\1", t)
    steps = []
    for _n, op, d in STEP_RE.findall(t):
        d = " ".join(d.split())
        if d:
            steps.append(((op or "").strip().upper(), d))
    steps = steps[:MAX_STEPS]
    if not steps:
        lines = [l.strip() for l in t.split("\n") if l.strip()][:MAX_STEPS]
        return {"plan": "\n".join(lines), "n_steps": 0, "format_ok": False,
                "operators": [], "ops_valid": False, "has_math": False}
    plan = "\n".join(f"Step {i} [{o}]: {d}" if o else f"Step {i}: {d}"
                     for i, (o, d) in enumerate(steps, 1))
    ops = [o for o, _ in steps]
    tagged = [o for o in ops if o]
    return {"plan": plan, "n_steps": len(steps),
            "format_ok": len(tagged) == len(ops) and bool(ops),
            "operators": ops,
            "ops_valid": bool(tagged) and all(o in OPS for o in tagged),
            "has_math": bool(ARITH_RE.search(plan))}


NUM_RE = re.compile(r"[-+]?\d[\d,]*(?:\.\d+)?|[-+]?\.\d+")
CALC_RE = re.compile(r"<<[^>]*>>")
BOXED_RE = re.compile(r"\\boxed\s*\{([^{}]*)\}")
HASH_RE = re.compile(r"####\s*([^\n]*)")
EQ_RE = re.compile(r"=\s*\$?\s*([-+]?\d[\d,]*(?:\.\d+)?)")
PHRASE_RE = re.compile(r"(?:final\s+answer|answer|result|therefore|so\s+the\s+answer)"
                       r"(?:\s*(?:\b(?:is|are|was|were|be|equals)\b|[:=]))*"
                       r"\s*\$?\s*([-+]?\d[\d,]*(?:\.\d+)?)", re.IGNORECASE)
RAMBLE = ("however", "wait", "but ", "but,", "let's correct", "let me correct",
          "there seems to be", "there is a mistake", "actually,", "on second thought",
          "hold on", "recalculat", "re-calculat", "let's re", "let me re")


def to_f(t):
    if t is None:
        return None
    c = str(t).strip().replace(",", "").replace("$", "").replace("%", "").replace("\\", "").rstrip(".")
    try:
        v = float(c)
    except ValueError:
        return None
    return v if math.isfinite(v) else None


def first_num(s):
    m = NUM_RE.search(s or "")
    return to_f(m.group(0)) if m else None


def ramble_cut(t):
    low, c = t.lower(), len(t)
    for m in RAMBLE:
        p = low.find(m)
        if p != -1:
            c = min(c, p)
    return c


def modal(t):
    vals = [to_f(m.group(0)) for m in NUM_RE.finditer(t)]
    vals = [v for v in vals if v is not None]
    if not vals:
        return None
    cnt = Counter(vals)
    best, bc = None, 1
    for v in vals:
        if cnt[v] > bc:
            best, bc = v, cnt[v]
    return (best, bc) if best is not None else None


def extract_answer(text):
    if not text:
        return None, "empty"
    c = CALC_RE.sub("", text)
    h = HASH_RE.findall(c)
    if h and first_num(h[-1]) is not None:
        return first_num(h[-1]), "hash"
    b = BOXED_RE.findall(c)
    if b and first_num(b[-1]) is not None:
        return first_num(b[-1]), "boxed"
    pre = c[:ramble_cut(c)]
    p = PHRASE_RE.findall(pre)
    if p and to_f(p[-1]) is not None:
        return to_f(p[-1]), "phrase_preramble"
    e = EQ_RE.findall(pre)
    if e and to_f(e[-1]) is not None:
        return to_f(e[-1]), "equals_preramble"
    m = modal(pre)
    if m and m[1] >= 2:
        return m[0], "modal_preramble"
    n = NUM_RE.findall(pre)
    if n and to_f(n[-1]) is not None:
        return to_f(n[-1]), "last_number_preramble"
    n = NUM_RE.findall(c)
    if n and to_f(n[-1]) is not None:
        return to_f(n[-1]), "last_number"
    return None, "none"


def extract_gold(a):
    return first_num(CALC_RE.sub("", a.split("####")[-1]))


def ok(p, g):
    return p is not None and g is not None and math.isclose(p, g, rel_tol=1e-4, abs_tol=1e-4)


# =============================================================================
# CSV (resume-safe)
# =============================================================================
FIELDS = ["idx", "question", "gold", "plan", "plan_steps", "plan_format_ok",
          "plan_operators", "plan_ops_valid", "plan_has_math",
          "guided_raw", "guided_pred", "guided_strategy", "guided_correct",
          "baseline_raw", "baseline_pred", "baseline_strategy", "baseline_correct",
          "error"]


def init_csv(p):
    os.makedirs(os.path.dirname(p), exist_ok=True)
    if not os.path.exists(p) or os.path.getsize(p) == 0:
        with open(p, "w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=FIELDS).writeheader()


def done_idx(p):
    if not os.path.exists(p):
        return set()
    d = set()
    with open(p, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            if r.get("idx", "").strip().isdigit() and not r.get("error"):
                d.add(int(r["idx"]))
    return d


def append_rows(p, rows):
    with open(p, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS, extrasaction="ignore")
        for r in rows:
            w.writerow({k: r.get(k, "") for k in FIELDS})
        f.flush()
        os.fsync(f.fileno())


# =============================================================================
# MAIN LOOP
# =============================================================================
init_csv(OUT_CSV)
test = load_dataset("openai/gsm8k", "main", split="test")
already = done_idx(OUT_CSV)
todo = [i for i in range(min(N_PROBLEMS, len(test))) if i not in already]
print(f"\n[run] {len(test)} in split | {len(already)} done | {len(todo)} to go")

stats = Counter()
strat = Counter()
t_start = time.time()

for bstart in range(0, len(todo), BATCH_SIZE):
    idxs = todo[bstart:bstart + BATCH_SIZE]
    qs = [" ".join(test[i]["question"].split()) for i in idxs]
    golds = [extract_gold(test[i]["answer"]) for i in idxs]
    rows = []
    try:
        # ---- Phase A ----
        plans_raw = gen_safe(planner, ptok,
                             [PROMPT_PLANNER.replace("{question}", q) for q in qs],
                             PLANNER_MAX_NEW)
        infos = [sanitise_plan(r) for r in plans_raw]

        # ---- Phase B ----
        eprompts = [PROMPT_EXECUTOR.replace("{question}", q)
                    .replace("{plan}", inf["plan"].strip() or FALLBACK_PLAN)
                    for q, inf in zip(qs, infos)]
        answers = [cut(a, EXEC_STOPS) for a in
                   gen_safe(executor, etok, eprompts, EXECUTOR_MAX_NEW, REPETITION_PENALTY)]

        bases = [""] * len(idxs)
        if RUN_BASELINE:
            bases = [cut(a, EXEC_STOPS) for a in
                     gen_safe(executor, etok,
                              [PROMPT_BASELINE.replace("{question}", q) for q in qs],
                              EXECUTOR_MAX_NEW, REPETITION_PENALTY)]

        for i, q, g, inf, ans, bse in zip(idxs, qs, golds, infos, answers, bases):
            pred, st = extract_answer(ans)
            gok = ok(pred, g)
            bpred, bst, bok = None, "", False
            if RUN_BASELINE:
                bpred, bst = extract_answer(bse)
                bok = ok(bpred, g)
            stats["n"] += 1
            stats["guided"] += int(gok)
            stats["baseline"] += int(bok)
            stats["fmt"] += int(inf["format_ok"])
            stats["opsok"] += int(inf["ops_valid"])
            stats["math"] += int(inf["has_math"])
            strat[st] += 1
            rows.append({"idx": i, "question": q, "gold": g, "plan": inf["plan"],
                         "plan_steps": inf["n_steps"], "plan_format_ok": inf["format_ok"],
                         "plan_operators": " ".join(inf["operators"]),
                         "plan_ops_valid": inf["ops_valid"], "plan_has_math": inf["has_math"],
                         "guided_raw": ans, "guided_pred": pred, "guided_strategy": st,
                         "guided_correct": gok, "baseline_raw": bse, "baseline_pred": bpred,
                         "baseline_strategy": bst, "baseline_correct": bok, "error": ""})
    except Exception as exc:
        traceback.print_exc(limit=2)
        for i, q, g in zip(idxs, qs, golds):
            rows.append({"idx": i, "question": q, "gold": g,
                         "error": f"{type(exc).__name__}: {exc}"})
        torch.cuda.empty_cache()

    append_rows(OUT_CSV, rows)

    done = stats["n"]
    if done:
        el = time.time() - t_start
        rate = el / done
        left = (len(todo) - done) * rate
        print(f"[{done}/{len(todo)}] acc={stats['guided']/done:.1%} "
              f"fmt={stats['fmt']/done:.0%} ops={stats['opsok']/done:.0%} "
              f"| {rate:.2f}s/q | ETA {left/3600:.2f}h", flush=True)

# =============================================================================
# SUMMARY
# =============================================================================
n = max(stats["n"], 1)
print("\n" + "=" * 70)
print(f"2-SLM GUIDED, n={stats['n']}")
print(f"  accuracy         : {stats['guided']}/{stats['n']} = {stats['guided']/n:.2%}")
if RUN_BASELINE:
    print(f"  baseline         : {stats['baseline']}/{stats['n']} = {stats['baseline']/n:.2%}")
    print(f"  delta            : {(stats['guided']-stats['baseline'])/n:+.2%}")
se = math.sqrt((stats['guided']/n) * (1 - stats['guided']/n) / n)
print(f"  95% CI           : [{stats['guided']/n-1.96*se:.2%}, {stats['guided']/n+1.96*se:.2%}]")
print(f"  plan format ok   : {stats['fmt']/n:.1%}")
print(f"  operators valid  : {stats['opsok']/n:.1%}")
print(f"  arithmetic leak  : {stats['math']/n:.1%}")
print("  parse strategies :")
for s, c in strat.most_common():
    print(f"     {s:<24} {c:>5} ({c/n:.1%})")
print(f"  wall clock       : {(time.time()-t_start)/3600:.2f} h")
print(f"  CSV              : {OUT_CSV}")
print("=" * 70)
print("\nDOWNLOAD THE CSV NOW: right panel > Data > /kaggle/working > download,")
print("or Save Version to persist it. Kaggle clears the session otherwise.")

[find] adapter -> /kaggle/input/datasets/tanishqborkar/sgft-planner/phaseA_planner
[cfg] planner base : Qwen/Qwen3-4B
[cfg] executor     : Qwen/Qwen2.5-Math-1.5B
[cfg] n            : 1319   batch=8   baseline=True

[load] planner base ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[load] planner adapter <- /kaggle/input/datasets/tanishqborkar/sgft-planner/phaseA_planner
[load] executor <- Qwen/Qwen2.5-Math-1.5B


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[vram] 2.04 GB allocated

[run] 1319 in split | 48 done | 1271 to go
[8/1271] acc=75.0% fmt=100% ops=100% | 7.56s/q | ETA 2.65h
[16/1271] acc=62.5% fmt=100% ops=100% | 8.05s/q | ETA 2.80h
[24/1271] acc=70.8% fmt=100% ops=100% | 7.84s/q | ETA 2.72h
[32/1271] acc=68.8% fmt=100% ops=97% | 7.88s/q | ETA 2.71h
[40/1271] acc=70.0% fmt=100% ops=98% | 7.82s/q | ETA 2.68h
[48/1271] acc=70.8% fmt=100% ops=98% | 8.24s/q | ETA 2.80h
[56/1271] acc=71.4% fmt=100% ops=98% | 8.30s/q | ETA 2.80h
[64/1271] acc=73.4% fmt=100% ops=98% | 8.83s/q | ETA 2.96h
[72/1271] acc=75.0% fmt=100% ops=99% | 9.27s/q | ETA 3.09h
[80/1271] acc=75.0% fmt=100% ops=99% | 9.10s/q | ETA 3.01h
[88/1271] acc=73.9% fmt=100% ops=99% | 9.13s/q | ETA 3.00h
[96/1271] acc=74.0% fmt=100% ops=99% | 9.27s/q | ETA 3.03h
[104/1271] acc=73.1% fmt=100% ops=99% | 9.53s/q | ETA 3.09h
[112/1271] acc=72.3% fmt=100% ops=99% | 9.76s/q | ETA 3.14h
[120/1271] acc=71.7% fmt=100% ops=98% | 9.90s/q | ETA 3.17h
[128/1271] acc=72.7% fmt=100% ops=98% | 1

In [5]:
# =============================================================================
# KAGGLE - EXECUTOR SWAP: does guidance help a NON-math-specialised executor?
# =============================================================================
# Drop-in replacement for your last script. Three differences:
#
#  1. MULTIPLE EXECUTORS, none math-specialised:
#       unsloth/Llama-3.2-1B  - different FAMILY entirely (the strong test)
#       Qwen/Qwen2.5-1.5B     - matched control: same family/size/tokenizer as
#                               Qwen2.5-Math-1.5B, differs ONLY in math training
#     Both ungated, both base-completion models (no chat template needed).
#
#  2. PLANS ARE REUSED from twoslm_full_1319.csv, so the 4B planner never runs
#     again. ~0.9h instead of ~3.8h, and every executor sees byte-identical
#     plans. Falls back to generating them if the CSV is missing.
#
#  3. PER-MODEL EOS. Your script hardcoded "<|endoftext|>" (Qwen). On a Llama
#     tokenizer that resolves to None, so generation never terminates and every
#     answer runs to the 640-token cap. Fixed below.
#
# PRECISION: keep USE_4BIT=False (fp16), matching the run that produced
# 71.60/72.07. Mixing 4-bit and fp16 numbers injects a ~6-point artifact larger
# than any effect you are measuring.
#
# A note on expectations: run this to find out what happens, not to confirm a
# hypothesis. A null here is still a result, and you already have two.
# =============================================================================

import os, re, csv, gc, json, math, time, glob, traceback
from collections import Counter
from typing import List, Optional, Tuple

import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_dataset

# =============================================================================
# CONFIG
# =============================================================================
EXECUTORS = [
    # different family, general purpose
    ("Qwen/Qwen2.5-1.5B",    "qwen2.5-1.5B"),    # matched control, general purpose
]

PLANNER_BASE = "Qwen/Qwen3-4B"      # only used if plans must be regenerated
ADAPTER_DIR  = None                  # auto-detected
PLANS_CSV    = None                  # auto-detected

N_PROBLEMS   = 1319
BATCH_SIZE   = 16                    # small executors -> bigger batches
USE_4BIT     = False                 # False = fp16. MUST match your reference run.
OUT_DIR      = "/kaggle/working"

PLANNER_MAX_NEW  = 192
EXECUTOR_MAX_NEW = 640
REPETITION_PENALTY = 1.15

# Reference numbers from your completed math-specialist run (fp16, n=1271)
REF_MATH = {"name": "Qwen2.5-Math-1.5B", "guided": 0.7160, "baseline": 0.7207}

PROMPT_PLANNER = ("Instruction: Provide a semantic plan without math.\n"
                  "Question: {question}\n"
                  "Plan:\n")
PROMPT_EXECUTOR = ("Solve the following math problem. A step-by-step plan is provided; follow it.\n"
                   "Reason step by step, and put your final answer within \\boxed{}.\n\n"
                   "Problem: {question}\n\n"
                   "Plan:\n{plan}\n\n"
                   "Solution:")
PROMPT_BASELINE = ("Solve the following math problem.\n"
                   "Reason step by step, and put your final answer within \\boxed{}.\n\n"
                   "Problem: {question}\n\n"
                   "Solution:")

assert "\\boxed" in PROMPT_EXECUTOR and "\\boxed" in PROMPT_BASELINE
assert "####" not in PROMPT_EXECUTOR
assert "{plan}" in PROMPT_EXECUTOR and "{plan}" not in PROMPT_BASELINE
assert PROMPT_PLANNER.endswith("Plan:\n")

EXEC_STOPS = ("\nProblem:", "\nQuestion:", "\nPlan:", "\nSolution:")
PLAN_STOPS = ("\nQuestion:", "\nInstruction:", "Instruction:")
OPS = {"ADD", "SUBTRACT", "MULTIPLY", "DIVIDE", "CONVERT", "RATIO", "COMPARE", "COUNT"}
MAX_STEPS = 9
FALLBACK_PLAN = ("Step 1 [COUNT]: Identify the quantities given in the problem.\n"
                 "Step 2 [COMPARE]: Determine the relationship between them.\n"
                 "Step 3 [ADD]: Compute the requested quantity.")


# =============================================================================
# TOKENIZER / MODEL LOADING  (per-model EOS - the key fix)
# =============================================================================
# Each family uses a different terminator. Hardcoding Qwen's breaks Llama.
#   Qwen2.5    <|endoftext|>        Llama-3.2  <|end_of_text|> / <|eot_id|>
#   Gemma-2    <eos>                TinyLlama  </s>
KNOWN_EOS = ["<|endoftext|>", "<|end_of_text|>", "<|eot_id|>", "<eos>", "</s>", "<|im_end|>"]


def prep_tok(tok):
    """Trust the tokenizer's own EOS; only supply a pad token if missing."""
    if tok.pad_token_id is None:
        if tok.eos_token is not None:
            tok.pad_token = tok.eos_token
        else:
            tok.add_special_tokens({"pad_token": "<|pad|>"})
    tok.padding_side = "left"        # REQUIRED for batched decoder-only generation
    return tok


def stop_ids(tok):
    """Every terminator this tokenizer actually knows about."""
    ids = set()
    if tok.eos_token_id is not None:
        ids.add(tok.eos_token_id)
    for t in KNOWN_EOS:
        i = tok.convert_tokens_to_ids(t)
        if isinstance(i, int) and i >= 0 and i != tok.unk_token_id:
            ids.add(i)
    ids.discard(None)
    assert ids, "no usable EOS - generation would never stop"
    return sorted(ids)


def load_lm(model_id):
    kw = {"device_map": "auto", "low_cpu_mem_usage": True, "trust_remote_code": True}
    if USE_4BIT:
        try:
            cfg = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
            pre = getattr(cfg, "quantization_config", None) is not None
        except Exception:
            pre = False
        if not pre:
            kw["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    for key in ["torch_dtype", "dtype"]:
        try:
            m = AutoModelForCausalLM.from_pretrained(model_id, **{**kw, key: torch.float16})
            m.eval()
            if getattr(m, "generation_config", None) is not None:
                m.generation_config.max_length = None
            return m
        except TypeError as e:
            if "dtype" not in str(e):
                raise
    raise RuntimeError(f"could not load {model_id}")


def load_exec(model_id):
    tok = prep_tok(AutoTokenizer.from_pretrained(model_id, trust_remote_code=True))
    m = load_lm(model_id)
    print(f"[load] {model_id} | eos={tok.eos_token!r}({tok.eos_token_id}) "
          f"pad={tok.pad_token!r}({tok.pad_token_id}) | stops={stop_ids(tok)}")
    return m, tok


# =============================================================================
# GENERATION
# =============================================================================
@torch.no_grad()
def gen_batch(model, tok, prompts, max_new, rep_pen=1.0):
    enc = tok(prompts, return_tensors="pt", padding=True,
              add_special_tokens=False).to(model.device)
    plen = enc["input_ids"].shape[1]
    out = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                         temperature=None, top_p=None, top_k=None,
                         repetition_penalty=rep_pen,
                         eos_token_id=stop_ids(tok), pad_token_id=tok.pad_token_id)
    return [t.strip() for t in tok.batch_decode(out[:, plen:], skip_special_tokens=True)]


def gen_safe(model, tok, prompts, max_new, rep_pen=1.0):
    try:
        return gen_batch(model, tok, prompts, max_new, rep_pen)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(prompts) == 1:
            return [""]
        h = len(prompts) // 2
        print(f"   [oom] splitting {len(prompts)} -> {h}+{len(prompts)-h}")
        return (gen_safe(model, tok, prompts[:h], max_new, rep_pen) +
                gen_safe(model, tok, prompts[h:], max_new, rep_pen))


# =============================================================================
# PLAN CLEANING + ANSWER EXTRACTION  (unchanged - identical scoring)
# =============================================================================
THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)
STEP_RE = re.compile(r"^[ \t]*Step[ \t]*(\d+)[ \t]*(?:\[[ \t]*([^\]]{1,32}?)[ \t]*\])?[ \t]*:[ \t]*(.*?)[ \t]*$",
                     re.MULTILINE | re.IGNORECASE)
STEP_SPLIT_RE = re.compile(r"[ \t]*(Step[ \t]*\d+[ \t]*(?:\[|:))", re.IGNORECASE)
ARITH_RE = re.compile(r"\d[ \t]*(?:[+*/×÷])[ \t]*\d|\d[ \t]+-[ \t]+\d|=[ \t]*\$?\d|<<[^>]*>>")
NUM_RE = re.compile(r"[-+]?\d[\d,]*(?:\.\d+)?|[-+]?\.\d+")
CALC_RE = re.compile(r"<<[^>]*>>")
BOXED_RE = re.compile(r"\\boxed\s*\{([^{}]*)\}")
HASH_RE = re.compile(r"####\s*([^\n]*)")
EQ_RE = re.compile(r"=\s*\$?\s*([-+]?\d[\d,]*(?:\.\d+)?)")
PHRASE_RE = re.compile(r"(?:final\s+answer|answer|result|therefore|so\s+the\s+answer)"
                       r"(?:\s*(?:\b(?:is|are|was|were|be|equals)\b|[:=]))*"
                       r"\s*\$?\s*([-+]?\d[\d,]*(?:\.\d+)?)", re.IGNORECASE)
RAMBLE = ("however", "wait", "but ", "but,", "let's correct", "let me correct",
          "there seems to be", "there is a mistake", "actually,", "on second thought",
          "hold on", "recalculat", "re-calculat", "let's re", "let me re")


def cut(text, markers):
    c = len(text)
    for m in markers:
        p = text.find(m)
        if p != -1:
            c = min(c, p)
    return text[:c].strip()


def sanitise_plan(raw):
    t = THINK_RE.sub("", raw or "")
    t = re.sub(r"^\s*<think>", "", t, flags=re.IGNORECASE)
    t = cut(t, PLAN_STOPS)
    t = re.sub(r"^\s*Plan\s*:\s*", "", t, flags=re.IGNORECASE)
    t = STEP_SPLIT_RE.sub(r"\n\1", t)
    steps = []
    for _n, op, d in STEP_RE.findall(t):
        d = " ".join(d.split())
        if d:
            steps.append(((op or "").strip().upper(), d))
    steps = steps[:MAX_STEPS]
    if not steps:
        lines = [l.strip() for l in t.split("\n") if l.strip()][:MAX_STEPS]
        return {"plan": "\n".join(lines), "n_steps": 0, "format_ok": False,
                "operators": [], "ops_valid": False, "has_math": False}
    plan = "\n".join(f"Step {i} [{o}]: {d}" if o else f"Step {i}: {d}"
                     for i, (o, d) in enumerate(steps, 1))
    ops = [o for o, _ in steps]
    tagged = [o for o in ops if o]
    return {"plan": plan, "n_steps": len(steps),
            "format_ok": len(tagged) == len(ops) and bool(ops), "operators": ops,
            "ops_valid": bool(tagged) and all(o in OPS for o in tagged),
            "has_math": bool(ARITH_RE.search(plan))}


def to_f(t):
    if t is None:
        return None
    c = str(t).strip().replace(",", "").replace("$", "").replace("%", "").replace("\\", "").rstrip(".")
    try:
        v = float(c)
    except ValueError:
        return None
    return v if math.isfinite(v) else None


def first_num(s):
    m = NUM_RE.search(s or "")
    return to_f(m.group(0)) if m else None


def ramble_cut(t):
    low, c = t.lower(), len(t)
    for m in RAMBLE:
        p = low.find(m)
        if p != -1:
            c = min(c, p)
    return c


def modal(t):
    vals = [v for v in (to_f(m.group(0)) for m in NUM_RE.finditer(t)) if v is not None]
    if not vals:
        return None
    cnt = Counter(vals)
    best, bc = None, 1
    for v in vals:
        if cnt[v] > bc:
            best, bc = v, cnt[v]
    return (best, bc) if best is not None else None


def extract_answer(text):
    if not text:
        return None, "empty"
    c = CALC_RE.sub("", text)
    h = HASH_RE.findall(c)
    if h and first_num(h[-1]) is not None:
        return first_num(h[-1]), "hash"
    b = BOXED_RE.findall(c)
    if b and first_num(b[-1]) is not None:
        return first_num(b[-1]), "boxed"
    pre = c[:ramble_cut(c)]
    p = PHRASE_RE.findall(pre)
    if p and to_f(p[-1]) is not None:
        return to_f(p[-1]), "phrase_preramble"
    e = EQ_RE.findall(pre)
    if e and to_f(e[-1]) is not None:
        return to_f(e[-1]), "equals_preramble"
    m = modal(pre)
    if m and m[1] >= 2:
        return m[0], "modal_preramble"
    n = NUM_RE.findall(pre)
    if n and to_f(n[-1]) is not None:
        return to_f(n[-1]), "last_number_preramble"
    n = NUM_RE.findall(c)
    if n and to_f(n[-1]) is not None:
        return to_f(n[-1]), "last_number"
    return None, "none"


def extract_gold(a):
    return first_num(CALC_RE.sub("", a.split("####")[-1]))


def ok(p, g):
    return p is not None and g is not None and math.isclose(p, g, rel_tol=1e-4, abs_tol=1e-4)


# =============================================================================
# GET THE PLANS: reuse from CSV, else regenerate once
# =============================================================================
test = load_dataset("openai/gsm8k", "main", split="test")
N = min(N_PROBLEMS, len(test))
QUESTIONS = [" ".join(test[i]["question"].split()) for i in range(N)]
GOLDS = [extract_gold(test[i]["answer"]) for i in range(N)]


def find_plans_csv():
    if PLANS_CSV and os.path.exists(PLANS_CSV):
        return PLANS_CSV
    for pat in ["/kaggle/working/twoslm_full_1319.csv",
                "/kaggle/input/**/twoslm_full_1319.csv",
                "/kaggle/input/**/*.csv"]:
        hits = sorted(glob.glob(pat, recursive=True))
        for h in hits:
            try:
                cols = next(csv.reader(open(h, newline="", encoding="utf-8")))
                if "plan" in cols and "idx" in cols:
                    return h
            except Exception:
                continue
    return None


PLANS = [None] * N
src = find_plans_csv()
if src:
    got = 0
    for r in csv.DictReader(open(src, newline="", encoding="utf-8")):
        if r.get("idx", "").strip().isdigit() and not r.get("error"):
            i = int(r["idx"])
            if 0 <= i < N and r.get("plan", "").strip():
                PLANS[i] = r["plan"]
                got += 1
    print(f"[plans] reused {got}/{N} from {src}")
missing = [i for i in range(N) if PLANS[i] is None]

if missing:
    print(f"[plans] regenerating {len(missing)} with the planner (this is the slow path)")
    ad = ADAPTER_DIR
    if not ad:
        hits = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
        if not hits:
            raise FileNotFoundError("No plans CSV and no adapter. Add one as a Dataset.")
        ad = os.path.dirname(hits[0])
    planner = PeftModel.from_pretrained(load_lm(PLANNER_BASE), ad, is_trainable=False).eval()
    try:
        ptok = prep_tok(AutoTokenizer.from_pretrained(ad, trust_remote_code=True))
    except Exception:
        ptok = prep_tok(AutoTokenizer.from_pretrained(PLANNER_BASE, trust_remote_code=True))
    for s in range(0, len(missing), BATCH_SIZE):
        ks = missing[s:s + BATCH_SIZE]
        raws = gen_safe(planner, ptok,
                        [PROMPT_PLANNER.replace("{question}", QUESTIONS[k]) for k in ks],
                        PLANNER_MAX_NEW)
        for k, raw in zip(ks, raws):
            PLANS[k] = sanitise_plan(raw)["plan"]
        print(f"   plans {s+len(ks)}/{len(missing)}", flush=True)
    del planner
    gc.collect()
    torch.cuda.empty_cache()

PLANS = [p.strip() if p and p.strip() else FALLBACK_PLAN for p in PLANS]
print(f"[plans] ready for {len(PLANS)} problems")


# =============================================================================
# RUN EACH EXECUTOR
# =============================================================================
FIELDS = ["idx", "question", "gold", "plan",
          "guided_raw", "guided_pred", "guided_strategy", "guided_correct",
          "baseline_raw", "baseline_pred", "baseline_strategy", "baseline_correct"]
RESULTS = {}

for model_id, tag in EXECUTORS:
    out_csv = os.path.join(OUT_DIR, f"exec_{tag}.csv")
    done = set()
    if os.path.exists(out_csv) and os.path.getsize(out_csv) > 0:
        done = {int(r["idx"]) for r in csv.DictReader(open(out_csv, newline=""))
                if r.get("idx", "").strip().isdigit()}
    else:
        with open(out_csv, "w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=FIELDS).writeheader()
    todo = [i for i in range(N) if i not in done]

    print("\n" + "=" * 78)
    print(f"EXECUTOR {model_id}   ({len(todo)} to run, {len(done)} done)")
    print("=" * 78)
    if not todo:
        continue

    model, tok = load_exec(model_id)
    st, strat = Counter(), Counter()
    t0 = time.time()

    for s in range(0, len(todo), BATCH_SIZE):
        ks = todo[s:s + BATCH_SIZE]
        qs, pls, gs = [QUESTIONS[k] for k in ks], [PLANS[k] for k in ks], [GOLDS[k] for k in ks]
        rows = []
        try:
            gtxt = [cut(a, EXEC_STOPS) for a in gen_safe(
                model, tok,
                [PROMPT_EXECUTOR.replace("{question}", q).replace("{plan}", p)
                 for q, p in zip(qs, pls)], EXECUTOR_MAX_NEW, REPETITION_PENALTY)]
            btxt = [cut(a, EXEC_STOPS) for a in gen_safe(
                model, tok,
                [PROMPT_BASELINE.replace("{question}", q) for q in qs],
                EXECUTOR_MAX_NEW, REPETITION_PENALTY)]
            for k, q, p, g, ga, ba in zip(ks, qs, pls, gs, gtxt, btxt):
                gp, gst = extract_answer(ga)
                bp, bst = extract_answer(ba)
                gok, bok = ok(gp, g), ok(bp, g)
                st["n"] += 1
                st["g"] += int(gok)
                st["b"] += int(bok)
                st["go"] += int(gok and not bok)
                st["bo"] += int(bok and not gok)
                strat[gst] += 1
                rows.append({"idx": k, "question": q, "gold": g, "plan": p,
                             "guided_raw": ga, "guided_pred": gp, "guided_strategy": gst,
                             "guided_correct": gok, "baseline_raw": ba, "baseline_pred": bp,
                             "baseline_strategy": bst, "baseline_correct": bok})
        except Exception as exc:
            traceback.print_exc(limit=1)
            torch.cuda.empty_cache()

        with open(out_csv, "a", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=FIELDS, extrasaction="ignore")
            for r in rows:
                w.writerow({c: r.get(c, "") for c in FIELDS})
            f.flush()
            os.fsync(f.fileno())

        if st["n"]:
            el = time.time() - t0
            print(f"[{st['n']}/{len(todo)}] guided={st['g']/st['n']:.1%} "
                  f"base={st['b']/st['n']:.1%} delta={(st['g']-st['b'])/st['n']:+.1%} "
                  f"| ETA {(len(todo)-st['n'])*el/st['n']/3600:.2f}h", flush=True)

    RESULTS[tag] = {"stats": dict(st), "strategies": dict(strat), "model": model_id}
    n = max(st["n"], 1)
    print(f"\n[{tag}] guided {st['g']}/{n} = {st['g']/n:.2%} | "
          f"baseline {st['b']}/{n} = {st['b']/n:.2%} | delta {(st['g']-st['b'])/n:+.2%}")
    print(f"[{tag}] strategy mix: {dict(strat.most_common(4))}")
    del model
    gc.collect()
    torch.cuda.empty_cache()


# =============================================================================
# THE INTERACTION TABLE
# =============================================================================
from math import comb


def mcnemar(b, c):
    n = b + c
    if n == 0:
        return 1.0
    return min(2 * sum(comb(n, i) for i in range(max(b, c), n + 1)) * 0.5 ** n, 1.0)


print("\n" + "=" * 92)
print("DOES GUIDANCE HELP?  BY EXECUTOR MATHEMATICAL SPECIALISATION")
print("=" * 92)
print(f"{'executor':<26}{'type':<16}{'baseline':>10}{'guided':>9}{'delta':>9}"
      f"{'g-only':>8}{'b-only':>8}{'p':>9}")
print("-" * 92)
print(f"{REF_MATH['name']:<26}{'math-spec.':<16}{REF_MATH['baseline']:>9.1%}"
      f"{REF_MATH['guided']:>9.1%}{REF_MATH['guided']-REF_MATH['baseline']:>+9.1%}"
      f"{'-':>8}{'-':>8}{'-':>9}")
for tag, r in RESULTS.items():
    s = r["stats"]
    n = max(s.get("n", 0), 1)
    p = mcnemar(s.get("go", 0), s.get("bo", 0))
    star = " *" if p < 0.05 else ""
    print(f"{tag:<26}{'general':<16}{s['b']/n:>9.1%}{s['g']/n:>9.1%}"
          f"{(s['g']-s['b'])/n:>+9.1%}{s.get('go',0):>8}{s.get('bo',0):>8}{p:>9.4f}{star}")
print("-" * 92)

json.dump(RESULTS, open(os.path.join(OUT_DIR, "executor_swap_summary.json"), "w"), indent=2)
print(f"\nsaved -> {OUT_DIR}/executor_swap_summary.json  and exec_*.csv")
print("""
READING IT
  A positive significant delta for the general models next to ~0 for the
  math-specialised one is an INTERACTION EFFECT: guidance helps executors that
  cannot already decompose problems. That one table explains your null result
  AND the base paper's positive results at once.

  If all three sit near zero, the finding is that solution guidance does not
  help at this scale regardless of executor. That contradicts the base paper
  and, with the controls you have, is worth reporting as-is.

  Quote the McNemar p, never the raw delta.

WATCH THE STRATEGY MIX
  Llama base models rarely emit \\boxed{} - that habit comes from math tuning.
  Expect equals_preramble to dominate for Llama. The parser handles it, but say
  so in the paper rather than letting a reviewer find that two executors were
  scored through different extraction paths.

DOWNLOAD THE CSVs before the session ends.
""")

[plans] reused 1319/1319 from /kaggle/working/twoslm_full_1319.csv
[plans] ready for 1319 problems

EXECUTOR Qwen/Qwen2.5-1.5B   (1319 to run, 0 done)


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

[load] Qwen/Qwen2.5-1.5B | eos='<|endoftext|>'(151643) pad='<|endoftext|>'(151643) | stops=[128247, 151643, 151645]
[16/1319] guided=50.0% base=37.5% delta=+12.5% | ETA 1.26h
[32/1319] guided=34.4% base=50.0% delta=-15.6% | ETA 1.24h
[48/1319] guided=41.7% base=50.0% delta=-8.3% | ETA 1.23h
[64/1319] guided=50.0% base=48.4% delta=+1.6% | ETA 1.24h
[80/1319] guided=55.0% base=50.0% delta=+5.0% | ETA 1.15h
[96/1319] guided=55.2% base=50.0% delta=+5.2% | ETA 1.13h
[112/1319] guided=55.4% base=46.4% delta=+8.9% | ETA 1.10h
[128/1319] guided=57.0% base=43.8% delta=+13.3% | ETA 1.09h
[144/1319] guided=57.6% base=45.8% delta=+11.8% | ETA 1.09h
[160/1319] guided=55.6% base=43.1% delta=+12.5% | ETA 1.10h
[176/1319] guided=54.0% base=43.2% delta=+10.8% | ETA 1.07h
[192/1319] guided=53.6% base=42.2% delta=+11.5% | ETA 1.05h
[208/1319] guided=52.9% base=43.3% delta=+9.6% | ETA 1.04h
[224/1319] guided=53.6% base=45.5% delta=+8.0% | ETA 1.03h
[240/1319] guided=54.2% base=46.7% delta=+7.5% | ETA 1.01